<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Exercise: Extend the Workflow </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

**Purpose:**  
Practice **coordination logic** by extending your two-agent system with a new task type and ensuring proper delegation.

**Exercise Task: Enhanced Budget Research Assistant**

Extend your existing two-agent system to handle **currency conversion and cost comparison**.

**New Requirements:**
- Add support for comparing the total budget with a **student budget limit** (e.g., 15,000 SAR)
- The Planner must recognize when currency conversion or comparison is needed
- The Executor must handle the new calculation correctly
- Inspect and improve the communication trace between agents

**Learning Objectives:**
- Handle additional task types
- Improve coordination between Planner and Executor
- Analyze communication logs for better delegation

---

## 1. Setup

In [1]:
!pip install -q --force-reinstall "langchain>=0.1.0,<1.0" "langchain-core==0.3.*" "langchain-openai==0.2.*" "langchain-community==0.3.*" langsmith duckduckgo-search ddgs faiss-cpu beautifulsoup4 requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 744.6/744.6 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18

In [2]:
import os
from getpass import getpass
from typing import Dict, List

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete!")

Enter your OpenAI API key: ··········
Setup complete!


In [3]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_react_agent, AgentExecutor
from langsmith import Client

client = Client()

search_tool = DuckDuckGoSearchRun(name="Web_Search")

@tool
def calculate_math(expression: str) -> str:
    """Perform mathematical calculations."""
    try:
        allowed = {"__builtins__": {}}
        return str(eval(expression, allowed, {}))
    except Exception as e:
        return f"Error: {str(e)}"

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
react_prompt = client.pull_prompt("hwchase17/react",
    dangerously_pull_public_prompt=True)

tools = [search_tool, calculate_math]

agent = create_react_agent(llm=llm, tools=tools, prompt=react_prompt)

executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=10,
    handle_parsing_errors=True
)

print("Tools, LLM, and Agent ready")

Tools, LLM, and Agent ready


## TODO: Update Role Instructions for Extended Task

In [4]:
# TODO 1: Update instructions to handle the new requirement (budget comparison)

planner_instruction = """You are a Planner Agent.
Your ONLY job is to break down the task into 3-4 short and focused steps.
Focus strictly on:
1. Researching flight price
2. Researching hotel price
3. Getting exchange rate
4. Calculating total cost and comparing with budget limit.

Do NOT include booking, packing, itinerary, daily expenses, or visa steps."""

executor_instruction = """You are an Executor Agent.
Follow the plan from the Planner exactly.
Use Web_Search for research and calculate_math for ALL calculations.
When a budget limit is given (e.g. 15,000 SAR), calculate the total cost in SAR and clearly state whether the trip is "Under Budget" or "Over Budget".
Give a clean, short final summary."""

print("Role instructions updated for budget comparison")

Role instructions updated for budget comparison


## TODO: Create / Update the Two Agents

In [5]:
from langchain_core.prompts import ChatPromptTemplate

# Planner Agent - Simple LLM chain (no tools)
planner_prompt = ChatPromptTemplate.from_template(
    """You are a Planner Agent. Break down the task into 3-4 clear, focused steps only.

Task: {input}

Plan must include research for flight, hotel, exchange rate, and final calculation + budget comparison."""
)

planner_chain = planner_prompt | llm

# Executor Agent
executor_prompt = react_prompt.partial(
    system_message=executor_instruction
)

executor_agent = create_react_agent(
    llm=llm,
    tools=[search_tool, calculate_math],
    prompt=executor_prompt
)

executor_executor = AgentExecutor(
    agent=executor_agent,
    tools=[search_tool, calculate_math],
    verbose=False,
    max_iterations=30,
    handle_parsing_errors=True
)

print("Planner and Executor Agents created!")

Planner and Executor Agents created!


## TODO: Extend the Message Passing Logic

In [6]:
def run_extended_two_agent_system(user_query: str):
    print("Starting Extended Two-Agent Budget Assistant\n")
    interactions = []#####

    # Step 1: Planner Agent
    print("Planner Agent: Creating plan...")
    plan_input = f"Create a short, focused step-by-step plan for this task: {user_query}"

    plan_result = planner_chain.invoke({"input": plan_input})
    plan_text = plan_result.content if hasattr(plan_result, "content") else str(plan_result)

    interactions.append({"from": "Planner", "to": "Executor", "message": plan_text})
    print(plan_text + "\n")

    # Step 2: Executor Agent
    print("Executor Agent: Executing the plan...")
    executor_input = f"""Follow this exact plan from the Planner:

{plan_text}

Original task: {user_query}

- Use Web_Search tool for any needed research
- Use calculate_math tool for every calculation
- If a budget limit (like 15,000 SAR) is mentioned, calculate total cost and clearly say "Under Budget" or "Over Budget"
- Give a clean and short final summary with all numbers in SAR."""

    executor_result = executor_executor.invoke({"input": executor_input})

    interactions.append({"from": "Executor", "to": "User", "message": "Extended task completed"})

    # Final Output
    print("\n" + "="*70)
    print("FINAL ANSWER:")
    print(executor_result["output"])

    print("\n" + "="*70)
    print("COMMUNICATION TRACE:")
    for i, msg in enumerate(interactions, 1):
        print(f"{i}. {msg['from']} → {msg['to']}")

    return executor_result["output"]

# Test the extended system
result = run_extended_two_agent_system(
    "Plan a budget trip from Riyadh to Istanbul for 7 days and check if the total cost is under 15,000 SAR"
)

Starting Extended Two-Agent Budget Assistant

Planner Agent: Creating plan...
### Step-by-Step Plan for Budget Trip from Riyadh to Istanbul

1. **Research Flights**: 
   - Use flight comparison websites (e.g., Skyscanner, Google Flights) to find the cheapest round-trip flights from Riyadh to Istanbul. 
   - Note the prices, airlines, and travel dates to ensure flexibility and the best deals.

2. **Find Accommodation**: 
   - Search for budget hotels or hostels in Istanbul on platforms like Booking.com or Airbnb. 
   - Filter results by price and location, and select options that fit your budget for 7 nights.

3. **Check Exchange Rates**: 
   - Look up the current exchange rate for Saudi Riyals (SAR) to Turkish Lira (TRY) using a reliable financial news source or currency converter. 
   - Calculate how much local currency you will need for daily expenses, including food, transportation, and activities.

4. **Calculate Total Cost and Compare**: 
   - Sum the costs of flights, accommodati

## Reflection Questions

1. How did you modify the Planner to handle the new budget comparison requirement?
2. Did the Executor correctly handle the additional calculation task?
3. What did you observe in the communication trace after extending the workflow?
4. What challenges appeared when adding a new task type?
5. How would you improve coordination if we add a third agent (e.g., Reviewer)?
